In [4]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def train_random_forest_with_ohe(
    file1="1500.csv",
    target_col="class",
    sep=",",
    test_size=0.2,
    random_state=42,
    n_estimators=50,
    max_depth=None,
    save_model_path="rf_pipeline1500.joblib"
):
    df = pd.read_csv(file1, sep=sep)

    if target_col not in df.columns:
        raise ValueError(f"Столбец {target_col!r} не найден в данных.")

    df = df.dropna(subset=[target_col]).copy()

    X = df.drop(columns=[target_col])
    y = df[target_col].copy()

    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

    if not numeric_cols and not categorical_cols:
        raise ValueError("После удаления target не осталось признаков.")

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    preprocessor_fitted = model.named_steps["preprocessor"]
    feature_names = preprocessor_fitted.get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_

    feature_importance = (
        pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    joblib.dump(model, save_model_path)

    return {
        "model": model,
        "accuracy": acc,
        "classification_report": report,
        "confusion_matrix": cm,
        "feature_importance": feature_importance,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "train_shape": X_train.shape,
        "test_shape": X_test.shape
    }


if __name__ == "__main__":
    result = train_random_forest_with_ohe(
        file1="1500.csv",
        target_col="class",
        sep=",",
        test_size=0.2,
        random_state=42,
        n_estimators=50,
        max_depth=None,
        save_model_path="rf_pipeline1500.joblib"
    )

    print("Accuracy:", result["accuracy"])
    print("Numeric columns:", result["numeric_cols"])
    print("Categorical columns:", result["categorical_cols"])
    print("Train shape:", result["train_shape"])
    print("Test shape:", result["test_shape"])
    print("\nClassification report:\n", result["classification_report"])
    print("\nConfusion matrix:\n", result["confusion_matrix"])
    print("\nTop feature importances:\n", result["feature_importance"].head(20))


Accuracy: 0.7913426645930534
Numeric columns: ['wave', 'intensity']
Categorical columns: ['brain_region']
Train shape: (910487, 3)
Test shape: (227622, 3)

Classification report:
               precision    recall  f1-score   support

     control       0.77      0.78      0.77     91035
        endo       0.76      0.78      0.77     45657
         exo       0.83      0.81      0.82     90930

    accuracy                           0.79    227622
   macro avg       0.79      0.79      0.79    227622
weighted avg       0.79      0.79      0.79    227622


Confusion matrix:
 [[71006  8239 11790]
 [ 7195 35508  2954]
 [14236  3081 73613]]

Top feature importances:
                       feature  importance
0              num__intensity    0.438683
1                   num__wave    0.342217
2    cat__brain_region_cortex    0.109655
3  cat__brain_region_striatum    0.109444
